<table>
  <tr>
    <td><div align="left"><font size="30">Gamma encoding</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Chapter 11

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`).
If you're running locally you need to have these packages installed.  If you're running on CoLab we have to first install machinevisiontoolbox which is not preinstalled, this will be a bit slow.

In [ ]:
# MVTB_BOOTSTRAP_CELL -- sets up the environment (Colab / JupyterLite / local install); click to expand. Generated from docs/notebooks/_mvtb_nb_bootstrap.py by sync_bootstrap.py -- do not hand-edit.

"""Environment bootstrap for machinevision-toolbox-python's Jupyter notebooks.

Installs the toolbox (and reports how) across the three environments a notebook in
this folder might run in: a local Jupyter/VS Code install, Google Colab, and
JupyterLite (Pyodide/WASM, in-browser).

This file is the single source of truth for that logic. Every notebook's own
bootstrap cell is a generated copy of this file's content, produced by
sync_bootstrap.py -- see docs/notebooks/README.md for the full explanation.
"""

import subprocess
import sys
from pathlib import Path


async def ensure_installed() -> bool:
    """Install machinevision-toolbox-python if needed, and report the environment.

    :returns: True if running on Google Colab, False otherwise.
    """
    if sys.platform == "emscripten":
        import micropip

        await micropip.install(
            [
                "opencv-python",
                "spatialmath-python",
                "pgraph-python",
                "ansitable",
                "mvtb-data",
                "tqdm",
                "requests",
                "ipywidgets",
            ]
        )
        import cv2  # noqa: F401 - force cv2 into module registry before toolbox import

        wheels = sorted(Path("/pypi").glob("machinevision_toolbox_python-*.whl"))
        if wheels:
            # Prefer the wheel bundled with this JupyterLite site. Relative,
            # not absolute: a leading slash resolves against the origin, not
            # this site's own base URL (a GitHub Pages project subpath).
            await micropip.install(f"pypi/{wheels[-1].name}", deps=False)
        else:
            # Fall back to PyPI when running outside the published site layout.
            await micropip.install("machinevision-toolbox-python", deps=False)
        where, colab = "in browser", False
    else:
        try:
            import google.colab  # noqa: F401
        except ImportError:
            where, colab = "locally", False
        else:
            print("Installing machinevision-toolbox-python...")
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "machinevision-toolbox-python",
                ],
                check=True,
            )
            where, colab = "on Colab", True

    import cv2

    import machinevisiontoolbox

    version = getattr(machinevisiontoolbox, "__version__", "unknown")
    opencv_version = getattr(cv2, "__version__", "unknown")
    print(f"Running {where} using MVTB v{version} with OpenCV {opencv_version}")
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

import numpy as np
from machinevisiontoolbox import *

# display result of assignments
if COLAB:
    %config ZMQInteractiveShell.ast_node_interactivity = 'last_expr_or_assign'
# make NumPy display a bit nicer
np.set_printoptions(linewidth=100, formatter={'float': lambda x: f"{x:10.4g}" if abs(x) > 1e-10 else f"{0:10.4g}"})

# Gamma

Gamma encoding and decoding is discussed in Section 10.3.6 of **Robotics, Vision & Control: Python**.  We will create an image of what a photographer calls a step wedge

In [ ]:
a = np.tile(np.repeat(np.arange(0, 1.1, 0.1), 10), (50, 1))
idisp(a);

Explore the pixel values with the cursor. The third bar has twice the pixel value of the second bar but it doesn't appear this way.  The last bar has twice the pixel value of the second last bar -- again the intensitites on the screen don't reflect that.

We need to remember that the pixel values go to a monitor which (by inherent physics or design) displays an intensity that is the pixel value to the power of ~2.2 -- we are seeing the pixel values squared!

Let's now display the square root of our image

In [ ]:
idisp(np.sqrt(a))


We see that the brightness steps now seem much more linear. This is only an approximation of the required gamma decoding but it shows the monitor non-linearity very clearly.

**The really important message is that the pixel values in an image files are the square-root of the intensities in the scene**.